# Notebook 02 — OFAC SDN List Loading

This notebook demonstrates loading the OFAC Specially Designated Nationals (SDN)
list and exploring the per-category name lists used in entity matching.

**What this covers:**
- Structure of the OFAC SDN CSV file
- `load_ofac()` function: splitting individuals vs. orgs/vessels/aircraft
- Exploring name distributions by entity type
- Why category silos matter for matching accuracy

**Why category silos matter:** Matching a person name against a list containing
vessel names introduces false positives.  By keeping `SANCTIONED_PERSON` and
`SANCTIONED_ORG` lists separate, we reduce noise and improve precision.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import io

## 1. OFAC SDN CSV Format

The OFAC SDN CSV downloaded from the US Treasury has the following key columns:

| Column | Description |
|--------|-------------|
| ent_num | Unique entity number |
| SDN_Name | Primary sanctioned name |
| SDN_Type | `individual`, `entity`, `vessel`, `aircraft` |
| Program | Sanctions program (e.g. SDGT, IRAN) |
| Title | Title/position (persons only) |
| Remarks | Additional notes |

Our mapping: `individual` → `SANCTIONED_PERSON`, everything else → `SANCTIONED_ORG`.

In [ ]:
# Create a mock OFAC SDN CSV for demonstration.
mock_csv_data = """ent_num,SDN_Name,SDN_Type,Program,Title,Call_Sign,Vess_type,Tonnage,GRT,Vess_flag,Vess_owner,Remarks
1,JOHN DOE,individual,SDGT,Mr.,,,,,,
2,JANE SMITH,individual,IRAN,Dr.,,,,,,
3,IVAN PETROV,individual,UKRAINE,,,,,,,
4,ALI HASSAN,individual,SDGT,,,,,,,
5,ACME CORP,entity,SDGT,,,,,,,
6,SHADOW TRADING LLC,entity,IRAN,,,,,,,
7,DARK STAR,vessel,CUBA,,DS001,Cargo,5000,4800,Panama,SHADOW TRADING LLC,
8,STEALTH BIRD,aircraft,SDGT,,N123,Cargo,,,,ALI HASSAN,
9,,individual,SDGT,,,,,,,
10,VALID ORG,entity,SDGT,,,,,,,
"""

mock_df = pd.read_csv(io.StringIO(mock_csv_data), dtype=str)
print(f"Total rows in mock CSV: {len(mock_df)}")
mock_df[['ent_num', 'SDN_Name', 'SDN_Type', 'Program']]

## 2. The `load_ofac()` Function

We'll write the mock CSV to a temp file and call `load_ofac()` on it.
The function:
1. Reads the CSV and validates required columns
2. Drops rows with null/empty `SDN_Name`
3. Splits rows by `SDN_Type`: `individual` → PERSON, others → ORG
4. Returns deduplicated sorted name lists

In [ ]:
import tempfile

# Write mock data to a temp file.
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, encoding='utf-8') as f:
    f.write(mock_csv_data)
    tmp_path = f.name

print(f"Mock CSV written to: {tmp_path}")

In [ ]:
from ner_ofac.ofac_loader import load_ofac

ofac_names = load_ofac(tmp_path)

print("=== SANCTIONED_PERSON ===")
for name in ofac_names['SANCTIONED_PERSON']:
    print(f"  - {name}")

print(f"\n=== SANCTIONED_ORG ===")
for name in ofac_names['SANCTIONED_ORG']:
    print(f"  - {name}")

## 3. Exploring the Name Lists

Understanding the distribution of name lengths and counts helps tune the
fuzzy and semantic matching parameters.

In [ ]:
import pandas as pd

for category, names in ofac_names.items():
    name_lengths = [len(n) for n in names]
    print(f"{category}:")
    print(f"  Count: {len(names)}")
    if name_lengths:
        print(f"  Avg name length: {sum(name_lengths)/len(name_lengths):.1f} chars")
        print(f"  Min/Max: {min(name_lengths)} / {max(name_lengths)} chars")
    print()

## 4. Category Silo Demonstration

The silo ensures that entity type from Comprehend determines which name list
is searched.  A `SANCTIONED_PERSON` entity only searches person names.
This prevents false matches like matching 'JOHN' against vessel name 'JOHN'S GLORY'.

In [ ]:
# Demonstrate that person names and org names are disjoint.
person_set = set(ofac_names['SANCTIONED_PERSON'])
org_set = set(ofac_names['SANCTIONED_ORG'])
overlap = person_set & org_set

print(f"Person names: {sorted(person_set)}")
print(f"Org names:    {sorted(org_set)}")
print(f"Overlap:      {overlap}  (should be empty for clean data)")

In [ ]:
# Clean up temp file.
import os
os.unlink(tmp_path)
print("Temp file removed.")

## Summary

- `load_ofac()` returns exactly two keys: `SANCTIONED_PERSON` and `SANCTIONED_ORG`
- Rows with null or empty `SDN_Name` are dropped (1 row in our mock data)
- Names are deduplicated and sorted for consistent indexing
- Category silos prevent cross-type false positives in matching